# DỰ ĐOÁN GIÁ CỔ PHIẾU VN-INDEX
## *Dựa trên biến động giá trong quá khứ — mã thử nghiệm: VCB*

**Môn:** Học Máy — HCMUTE  
**Sinh viên thực hiện:** Bá Hoài Sơn — Bùi Thanh Tú  
**Dữ liệu:** Yahoo Finance qua thư viện `yfinance`

## 1. Bài toán

- Dự đoán **biến động giá cổ phiếu** chỉ từ chuỗi giá quá khứ.
- **Mã thử nghiệm:** VCB (Vietcombank — `VCB.VN`).
- **Loại bài toán:** Hồi quy có giám sát trên chuỗi thời gian.
- **Ý nghĩa:** Hỗ trợ đầu tư swing trading, kiểm chứng EMH.

## 2. Dữ liệu

- **Nguồn:** Yahoo Finance qua `yfinance` → `data/vcb_stock.csv`.
- **Quy mô:** 7.673 quan sát (4.209 phiên ngày + 3.464 phiên giờ).
- **Khoảng:** 2009-06-30 → 2026-05-15.
- **Cột:** `Date`, OHLCV, `Interval`, `Ticker`/`Symbol`.

## 3. Lịch sử giá VCB

![](../image/price_history_daily.png)

## 4. Cạm bẫy ban đầu

| Thử nghiệm | DirAcc | Vấn đề |
|------------|-------:|--------|
| Dự đoán **giá tuyệt đối** | — | LR sao chép giá; RF/KNN R² âm vì không ngoại suy được |
| Dự đoán **return 1 ngày** | 43-44 % | Nhiễu áp đảo + 10 % phiên có Return = 0 |
| Dự đoán **log-return 20 ngày** | **54.8 %** | ✓ Vượt ngẫu nhiên |

→ **Bài học**: chọn đúng target và horizon là chìa khóa.

## 5. Giải pháp: dự đoán LOG-RETURN 20 phiên

$$\hat r_{t+20} = f(\mathbf x_t),\qquad \hat C_{t+20} = C_t\,e^{\hat r_{t+20}}$$

- **20 phiên** ≈ 1 tháng giao dịch — phổ biến trong swing trading.
- **Log-return** symmetric + cộng được + gần Gaussian.
- Target stationary → mô hình phi tuyến tổng quát hóa công bằng.

## 6. 25 đặc trưng stationary

| Nhóm | Đặc trưng |
|---|---|
| Momentum | `Return_{1,2,3,5,10,20,60}` |
| Xu hướng | `MA{5,10,20,50,100}_Ratio` |
| Volatility | `Vol_{5,10,20}` |
| Chỉ báo | `RSI_14`, `MACD`, `MACD_Signal`, `MACD_Hist`, `Bollinger_b` |
| Biên độ | `HL_Range`, `OC_Range` |
| Volume | `Vol_Change`, `Volume_MA20_Ratio` |
| Trend dài | `Trend_MA50_200` (sign MA50 - MA200) |

## 7. Bốn mô hình thử nghiệm

| Mô hình | Vai trò |
|---------|---------|
| **Linear Regression** | Baseline tuyến tính |
| **KNN (k=25)** | Phi tham số, học theo phiên giống |
| **Random Forest (500 cây)** | Phi tuyến, robust |
| **Voting Ensemble** | Trung bình 3 mô hình trên |

*Pipeline(StandardScaler + estimator) — tránh leak + công bằng.*

## 8. Quy trình & đánh giá

1. Load (`load_raw_data`).
2. Feature engineering (`build_features`, h=20).
3. Chronological split 80/20.
4. Train 4 mô hình.
5. Đánh giá: **MAE, RMSE, R², DirAcc, DirAcc_filt** trên log-return + **MAPE** trên giá.
6. Sweep đa horizon {1, 5, 10, 20, 60}.

In [1]:
import sys
sys.path.insert(0, '../scripts')
from ml_utils import run_full_pipeline

result = run_full_pipeline(horizon=20, save_images=False)
print('--- METRIC LOG-RETURN (horizon=20) ---')
display(result['metrics_return'])
print('--- METRIC GIÁ (VND) ---')
display(result['metrics_price'])

--- METRIC LOG-RETURN (horizon=20) ---


,MAE,RMSE,R2,DirAcc(%),DirAcc_filt(%)
Model,,,,,
Linear Regression,0.0455,0.0650,-0.1167,54.7619,54.3175
KNN,0.0489,0.0673,-0.1940,50.8772,50.9749
Random Forest,0.0449,0.0649,-0.1132,52.3810,51.8106
Ensemble,0.0450,0.0647,-0.1036,52.7569,52.7855


--- METRIC GIÁ (VND) ---


,MAE (VND),RMSE (VND),MAPE(%)
Model,,,
Linear Regression,2752.88,4036.32,4.54
KNN,2967.66,4171.20,4.91
Random Forest,2715.23,4031.34,4.47
Ensemble,2727.60,4016.44,4.50


## 9. So sánh 4 mô hình (horizon = 20)

![](../image/model_comparison.png)

- **Tất cả 4 mô hình vượt 50 %.**
- **Linear Regression dẫn đầu DirAcc 54.76 %** — bài học: đơn giản không có nghĩa là yếu.

## 10. Dự đoán vs Thực tế (giá, h=20)

![](../image/predictions_vs_actual.png)

MAPE giá ~ 4.5 % cho horizon 1 tháng.

## 11. SWEEP ĐA HORIZON — phát hiện chính

![](../image/horizon_comparison.png)

| h | Best | DirAcc |
|--:|------|-------:|
| 1 | KNN | 50.7 % |
| 5 | LR | 48.7 % |
| 10 | RF | 50.6 % |
| **20** | **LR** | **54.8 %** |
| **60** | **Ensemble** | **61.6 %** |

→ Tín hiệu xu hướng **TĂNG THEO HORIZON**.

## 12. Hiệu Quả Thị Trường (EMH)

- DirAcc 1 ngày ≈ 50 % → **xác nhận EMH dạng yếu** ở ngắn hạn.
- DirAcc 20-60 ngày = 55-62 % → tín hiệu trung-dài hạn **dự đoán được**.
- Khoảng cách 4.8 - 11.6 điểm so với 50 %: ý nghĩa thống kê 2.7-6.5σ.

**Lưu ý:** Dự đoán đúng chiều ≠ kiếm lợi sau phí giao dịch.

## 13. Hạn chế & Hướng phát triển

**Hạn chế:**
- Chỉ dùng giá quá khứ, một mã (VCB), một lần chia 80/20.
- Siêu tham số chọn theo kinh nghiệm.

**Hướng phát triển:**
- Walk-forward validation + bootstrap CI.
- Sentiment + biến vĩ mô.
- LSTM / Transformer cho chuỗi thời gian.
- Mở rộng rổ VN30, tích hợp chi phí giao dịch.

## 14. Cảm ơn — Q & A

**Bá Hoài Sơn — Bùi Thanh Tú**  
*Học Máy — HCMUTE*

- Mã nguồn: `scripts/ml_utils.py`
- Tái lập: `python scripts/fetch_data.py` → mở 4 notebook
- Phát hiện chính: **DirAcc 54.8 %** ở horizon = 20